# Training Spike - SEA-LION v3 9B

**Purpose: de-risk the stack before Phase 3 builds 800-1500 examples.**

Base model is locked to `aisingapore/Gemma-SEA-LION-v3-9B-IT` (Phase 0, re-confirmed
2026-09-02: E2B's tokenizer advantage doesn't survive a bf16-less T4 - Unsloth forces
fp32 for gemma4, so E2B loaded *larger* than 9B in 4-bit, 7.45GB vs 6.16GB). This
notebook no longer compares models; it only verifies the stack on 9B:

1. Does it load in 4-bit on a free T4?
2. Does Unsloth take LoRA steps on it?
3. How much VRAM, how long per step?
4. Does it emit coherent, grounded **Burmese** after 30 steps?

Throwaway 18-example seed set. NOT the Phase 3 dataset.

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install unsloth
!pip -q install --no-deps --upgrade peft accelerate bitsandbytes

## Seed data

The repo is **private**, so the raw GitHub URL will 404. The cell below tries it anyway (in case you make the repo public) and otherwise prompts you to upload `data/samples/spike_seed.jsonl` from your machine.

In [ ]:
import json, os, urllib.request, time, gc, torch

# Prefer a local spike_seed.jsonl if one is already in the working directory.
# Otherwise the repo is PRIVATE, so raw.githubusercontent.com 404s without a
# token and we fall back to an interactive upload.
URL = ("https://raw.githubusercontent.com/Thant9330/EPS-Burmese-Assistant/"
       "master/data/samples/spike_seed.jsonl")

raw = None
if os.path.exists("spike_seed.jsonl"):
    raw = open("spike_seed.jsonl", encoding="utf-8").read()
    print("loaded from local spike_seed.jsonl")
else:
    try:
        raw = urllib.request.urlopen(URL, timeout=20).read().decode()
        print("loaded from GitHub raw URL")
    except Exception as e:
        print("raw URL unavailable (", type(e).__name__, ") - falling back to upload")
        print("Pick data/samples/spike_seed.jsonl from your machine:")
        from google.colab import files
        up = files.upload()
        raw = list(up.values())[0].decode("utf-8")

rows = [json.loads(l) for l in raw.splitlines() if l.strip()]
print(len(rows), "examples |",
      "grounded=", sum(r["kind"] == "grounded" for r in rows),
      "refusal=", sum(r["kind"] == "refusal" for r in rows))

# Hold two out (one grounded, one refusal) so BEFORE/AFTER is on unseen questions.
HELD = [rows[2], rows[-1]]
TRAIN = [r for r in rows if r not in HELD]
print("train=", len(TRAIN), " held_out=", len(HELD))
print("held-out Q1:", HELD[0]["question"], "|", HELD[0]["kind"])
print("held-out Q2:", HELD[1]["question"], "|", HELD[1]["kind"])

## The experiment

Same data, same LoRA config, same steps - only the base model changes.
Each model is wrapped in try/except so one failure still leaves the other's numbers.

In [ ]:
MODELS = [
    ("aisingapore/Gemma-SEA-LION-v3-9B-IT", "9B, gemma2, 3.32x Burmese - locked base model"),
]
MAX_SEQ, STEPS = 2048, 30
results = {}

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# T4 has no bfloat16. Leaving this to autocast gives
# "self and mat2 must have the same dtype, but got Half and BFloat16".
BF16 = torch.cuda.is_bf16_supported()
FP16 = not BF16
print("bf16 supported:", BF16)


def prompt_of(r):
    return (r["system"] + "\n\n### Context\n" + r["context"]
            + "\n\n### Question\n" + r["question"])


def msg(role, text, parts):
    return {"role": role,
            "content": [{"type": "text", "text": text}] if parts else text}


def needs_parts(tok):
    """Gemma 4 is multimodal: with tokenize=True its processor iterates content
    as typed parts and blows up on a bare string. Gemma 2 wants the bare string.
    Probe with the REAL call signature - a tokenize=False probe passes for both
    and tells you nothing."""
    for parts in (False, True):
        try:
            tok.apply_chat_template([msg("user", "hi", parts)], tokenize=True,
                                    add_generation_prompt=True, return_tensors="pt")
            return parts
        except Exception:
            continue
    raise RuntimeError("neither content format works for this tokenizer")


def build_text(tok, r, parts):
    return tok.apply_chat_template(
        [msg("user", prompt_of(r), parts),
         msg("assistant", r["answer"], parts)], tokenize=False)


def generate(model, tok, r, parts, n=220):
    FastLanguageModel.for_inference(model)
    # Render to text first, then tokenize with add_special_tokens=False. Going
    # straight to tokenize=True double-BOSes (the template emits BOS, the
    # tokenizer adds another) and returns no attention_mask, which made the 9B
    # emit EOS immediately and return an empty string.
    text = tok.apply_chat_template([msg("user", prompt_of(r), parts)],
                                   tokenize=False, add_generation_prompt=True)
    # NOTE: text= must be a KEYWORD. Gemma 4 loads a *processor*, whose first
    # positional arg is images, so tok(text, ...) silently becomes
    # images=<prompt>, text=None -> TypeError inside processing_gemma4.py.
    enc = tok(text=text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**enc, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


def run(name, note):
    print("=" * 78)
    print(name, "|", note)
    print("=" * 78, flush=True)
    rec = {"note": note}

    t0 = time.time()
    model, tok = FastLanguageModel.from_pretrained(
        model_name=name, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)
    rec["load_s"] = round(time.time() - t0, 1)
    rec["vram_load_GB"] = round(torch.cuda.memory_allocated() / 1e9, 2)
    parts = needs_parts(tok)
    rec["content_parts"] = parts
    print("loaded in", rec["load_s"], "s | VRAM", rec["vram_load_GB"],
          "GB | content_parts =", parts, flush=True)

    rec["before"] = [generate(model, tok, h, parts) for h in HELD]
    print("--- BEFORE training ---")
    for i, b in enumerate(rec["before"]):
        print("[", i, "]", repr(b[:400]) if not b.strip() else b[:400])

    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)

    ds = Dataset.from_list([{"text": build_text(tok, r, parts)} for r in TRAIN])
    trainer = SFTTrainer(
        model=model, tokenizer=tok, train_dataset=ds,
        args=SFTConfig(per_device_train_batch_size=1, gradient_accumulation_steps=4,
                       warmup_steps=5, max_steps=STEPS, learning_rate=2e-4,
                       logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
                       lr_scheduler_type="linear", seed=0, output_dir="out",
                       report_to="none", max_length=MAX_SEQ,
                       fp16=FP16, bf16=BF16,
                       dataset_text_field="text"))

    FastLanguageModel.for_training(model)
    t1 = time.time()
    stats = trainer.train()
    rec["train_s"] = round(time.time() - t1, 1)
    rec["s_per_step"] = round(rec["train_s"] / STEPS, 2)
    rec["final_loss"] = round(stats.training_loss, 4)
    rec["peak_vram_GB"] = round(torch.cuda.max_memory_reserved() / 1e9, 2)
    print("trained", STEPS, "steps in", rec["train_s"], "s |",
          rec["s_per_step"], "s/step | loss", rec["final_loss"],
          "| peak", rec["peak_vram_GB"], "GB", flush=True)

    rec["after"] = [generate(model, tok, h, parts) for h in HELD]
    print("--- AFTER training ---")
    for i, a in enumerate(rec["after"]):
        print("[", i, "]", repr(a[:400]) if not a.strip() else a[:400])

    del model, trainer
    return rec


for _name, _note in MODELS:
    err = None
    try:
        results[_name] = run(_name, _note)
    except Exception as e:
        # Only record the message here. Freeing VRAM inside this block is
        # useless: `e` holds the traceback, which holds run()'s frame, which
        # still references the model. Python drops `e` when the block exits.
        err = type(e).__name__ + ": " + str(e)

    if err is not None:
        results[_name] = {"note": _note, "ERROR": err}
        print("!!! FAILED:", err)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(">>> VRAM after cleanup:",
          round(torch.cuda.memory_allocated() / 1e9, 2), "GB allocated,",
          round(torch.cuda.memory_reserved() / 1e9, 2), "GB reserved", flush=True)

## Comparison - this is what decides the base model

In [ ]:
hdr = "%-42s %6s %7s %8s %7s" % ("model", "load", "s/step", "peakGB", "loss")
print(hdr)
print("-" * len(hdr))
for k, v in results.items():
    short = k.split("/")[-1]
    if "ERROR" in v:
        print("%-42s  FAILED: %s" % (short, v["ERROR"][:30]))
    else:
        print("%-42s %6s %7s %8s %7s" % (short, v["load_s"], v["s_per_step"],
                                         v["peak_vram_GB"], v["final_loss"]))

print("")
print("BURMESE QUALITY - judge yourself; no metric replaces a native reader:")
for k, v in results.items():
    if "ERROR" in v:
        continue
    print("")
    print("#" * 78)
    print("#", k)
    print("#" * 78)
    for i in range(len(HELD)):
        print("")
        print("--- held-out Q%d (%s): %s" % (i + 1, HELD[i]["kind"], HELD[i]["question"]))
        print("  BEFORE:", v["before"][i][:500])
        print("  AFTER :", v["after"][i][:500])

print("")
print("JSON to paste back into the repo:")
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk not in ("before", "after")}
                  for k, v in results.items()}, indent=2))

## What to look for

**Stack works** - loads in 4-bit, takes steps, peak VRAM comfortably under 15GB.

**Burmese quality** - compare BEFORE vs AFTER:
- Is the Burmese grammatical and natural?
- Are Korean terms (고용센터, 외국인등록) preserved rather than mangled?
- Does it stay grounded in the context instead of inventing rules?
- Does the held-out refusal example actually refuse?

If all four hold after 30 steps on 9B, the stack is de-risked - move to Phase 3.

Paste the JSON block back to Claude to have it written into the repo.